In [1]:
# Install required packages
%pip install pandas numpy pyyaml torch -q

import pandas as pd
import numpy as np


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
# Generate a sample dataset with realistic data quality issues
np.random.seed(42)
n_rows = 100

df = pd.DataFrame({
    "customer_id": [i if np.random.random() > 0.1 else None for i in range(1, n_rows + 1)],
    "order_date": [f"2024-{np.random.randint(1,13):02d}-{np.random.randint(1,29):02d}" 
                   if np.random.random() > 0.15 else None for _ in range(n_rows)],
    "price": np.random.uniform(10, 500, n_rows).round(2),
    "quantity": np.random.randint(1, 10, n_rows),
    "product_name": [f"Product_{chr(65 + i % 26)}" for i in range(n_rows)],
    "notes": [None] * n_rows,  # High null column (100% null)
    "discount": [np.random.uniform(0, 0.3) if np.random.random() > 0.95 else None for _ in range(n_rows)]  # 95% null
})

print(f"Generated dataset with {len(df)} rows and {len(df.columns)} columns")
df.head(10)

Generated dataset with 100 rows and 7 columns


,customer_id,order_date,price,quantity,product_name,notes,discount
0,1.0,NaN,485.07,3,Product_A,None,NaN
1,2.0,2024-04-04,360.15,1,Product_B,None,NaN
2,3.0,2024-07-07,30.12,4,Product_C,None,NaN
3,4.0,2024-11-04,205.42,9,Product_D,None,NaN
4,5.0,2024-07-27,222.43,3,Product_E,None,NaN
5,6.0,2024-12-02,374.58,9,Product_F,None,NaN
6,NaN,2024-09-21,132.92,7,Product_G,None,NaN
7,8.0,2024-12-12,100.32,4,Product_H,None,NaN
8,9.0,2024-11-26,49.63,3,Product_I,None,NaN
9,10.0,NaN,219.87,5,Product_J,None,NaN


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   customer_id   87 non-null     float64
 1   order_date    85 non-null     str    
 2   price         100 non-null    float64
 3   quantity      100 non-null    int32  
 4   product_name  100 non-null    str    
 5   notes         0 non-null      object 
 6   discount      4 non-null      float64
dtypes: float64(3), int32(1), object(1), str(2)
memory usage: 5.2+ KB


In [3]:
# Import the neuro-symbolic agent
import importlib.util
import sys

spec = importlib.util.spec_from_file_location("neuro_symbolic_agent2", "neuro_symbolic_agent2.py")
neuro_symbolic_agent2 = importlib.util.module_from_spec(spec)
sys.modules["neuro_symbolic_agent2"] = neuro_symbolic_agent2
spec.loader.exec_module(neuro_symbolic_agent2)

run_neuro_symbolic_agent = neuro_symbolic_agent2.run_neuro_symbolic_agent
print("Neuro-symbolic agent loaded successfully!")

Neuro-symbolic agent loaded successfully!


In [4]:
data_context = {
    "null_ratio": df.isna().mean().mean(),
    "row_count": len(df),
    "column_importance": 0.8,
    "historical_failure_rate": 0.2
}

In [5]:
clean_df, decisions = run_neuro_symbolic_agent(df, data_context)

In [6]:
print("\n--- Rule Decisions ---")
for d in decisions:
    print(d)



--- Rule Decisions ---
{'rule_id': 'R001', 'rule_name': 'Enforce required columns', 'base_action': 'fail_pipeline', 'confidence': 0.548, 'applied': False, 'reason': 'Skipped fail_pipeline (safe execution mode)'}
{'rule_id': 'R002', 'rule_name': 'Drop high-null columns', 'base_action': 'drop_column', 'confidence': 0.547, 'applied': True, 'reason': 'Applied by agent ranking'}
{'rule_id': 'R003', 'rule_name': 'Drop rows with critical nulls', 'base_action': 'drop_row', 'confidence': 0.547, 'applied': True, 'reason': 'Applied by agent ranking'}
{'rule_id': 'R004', 'rule_name': 'Normalize dates', 'base_action': 'normalize_date', 'confidence': 0.547, 'applied': True, 'reason': 'Applied by agent ranking'}
{'rule_id': 'R005', 'rule_name': 'Create total_amount', 'base_action': 'derive_column', 'confidence': 0.547, 'applied': False, 'reason': 'Candidate rule'}


In [7]:
print("\n--- Dataset Info After Agent Execution ---")
print(clean_df.info())


--- Dataset Info After Agent Execution ---
<class 'pandas.DataFrame'>
Index: 76 entries, 1 to 97
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   customer_id   76 non-null     float64       
 1   order_date    76 non-null     datetime64[us]
 2   price         76 non-null     float64       
 3   quantity      76 non-null     int32         
 4   product_name  76 non-null     str           
dtypes: datetime64[us](1), float64(2), int32(1), str(1)
memory usage: 3.3 KB
None


In [ ]:
clean_df.to_csv("cleaned_dataset.csv", index=False)